# Elastic Green's functions

The Kelvin fundamental solution — the displacement produced by a point force
in an infinite isotropic medium — in plane strain and in three dimensions,
built symbolically and checked against its closed form. The second-gradient
object $\mathbb{\Gamma}=-\mathrm{HESS}(\boldsymbol{G})$, symmetrized, is the
kernel every micromechanical Green operator is built from.

This exercises the differential operators of
Curvilinear differential calculus on a genuinely
non-trivial field: a closed-form identity that only holds if every Christoffel
term is right. Background on the elastic Green operator: [mura1987](@cite).

In [1]:
using TensND
using LinearAlgebra
using SymPy
using Tensors

## Plane strain (2-D)

In polar coordinates, with $\nu$ the Poisson ratio and $\mu$ the shear
modulus:

$$
\boldsymbol{G}=\frac{1}{8\pi\mu(1-\nu)}
\Bigl(\underline{e}^r\otimes\underline{e}^r-(3-4\nu)\log r\;\boldsymbol{1}\Bigr)
$$

In [2]:
Polar = coorsys_polar()
r, θ = getcoords(Polar)
𝐞ʳ, 𝐞ᶿ = unitvec(Polar)
@set_coorsys Polar
ℬᵖ = normalized_basis(Polar)

𝕀₂, 𝕁₂, 𝕂₂ = iso_projectors(Val(2), Val(Sym))
𝟏₂ = tens_Id2(Val(2), Val(Sym))

E = symbols("E", positive = true)
ν = symbols("ν", real = true)
k = E / (3(1 - 2ν))
μ = E / (2(1 + ν))
λ = k - 2μ / 3

𝐆 = tsimplify(1 / (8 * PI * μ * (1 - ν)) * (𝐞ʳ ⊗ 𝐞ʳ - (3 - 4ν) * log(r) * 𝟏₂))

2×2 TensND.TensRotated{2, 2, Sym{PyCall.PyObject}, Tensors.SymmetricTensor{2, 2, Sym{PyCall.PyObject}, 3}}:
 (ν + 1)*(-(4*ν - 3)*log(r) - 1)/(4*pi*E*(ν - 1))  …                                         0
                                                0     (-4*ν^2 - ν + 3)*log(r)/(4*pi*E*(ν - 1))

## The Green operator $\mathbb{\Gamma}$

$\mathbb{\Gamma}=-\mathrm{HESS}(\boldsymbol{G})$, symmetrized over both index
pairs so that it acts on symmetric strain tensors:

In [3]:
HG = -tsimplify(HESS(𝐆))
aHG = get_array(HG)
𝕄 = SymmetricTensor{4, 2}((i, j, k, l) -> (aHG[i, k, j, l] + aHG[j, k, i, l] + aHG[i, l, j, k] + aHG[j, l, i, k]) / 4)
ℾ = tsimplify(Tens(𝕄, ℬᵖ))

2×2×2×2 TensND.TensRotated{4, 2, Sym{PyCall.PyObject}, Tensors.SymmetricTensor{4, 2, Sym{PyCall.PyObject}, 9}}:
[:, :, 1, 1] =
 (-4*ν^2 - ν + 3)/(4*pi*E*r^2*(ν - 1))                              0
                                     0  (-ν - 1)/(4*pi*E*r^2*(ν - 1))

[:, :, 2, 1] =
                             0  (-ν - 1)/(4*pi*E*r^2*(ν - 1))
 (-ν - 1)/(4*pi*E*r^2*(ν - 1))                              0

[:, :, 1, 2] =
                             0  (-ν - 1)/(4*pi*E*r^2*(ν - 1))
 (-ν - 1)/(4*pi*E*r^2*(ν - 1))                              0

[:, :, 2, 2] =
 (-ν - 1)/(4*pi*E*r^2*(ν - 1))                                       0
                             0  (4*ν^2 + 3*ν - 1)/(4*pi*E*r^2*(ν - 1))

The closed form it must reproduce:

$$
\mathbb{\Gamma}=\frac{1}{8\pi\mu(1-\nu)r^{2}}
\Bigl(-2\mathbb{J}+2(1-2\nu)\mathbb{I}
+2(\boldsymbol{1}\otimes\underline{e}^r\otimes\underline{e}^r
  +\underline{e}^r\otimes\underline{e}^r\otimes\boldsymbol{1})
+8\nu\,\underline{e}^r\stackrel{s}{\otimes}\boldsymbol{1}\stackrel{s}{\otimes}\underline{e}^r
-8\,\underline{e}^r{}^{\otimes4}\Bigr)
$$

In [4]:
ℾ₂ = tsimplify(
    1 / (8PI * μ * (1 - ν) * r^2) * (
        -2𝕁₂ + 2(1 - 2ν) * 𝕀₂ + 2(𝟏₂ ⊗ 𝐞ʳ ⊗ 𝐞ʳ + 𝐞ʳ ⊗ 𝐞ʳ ⊗ 𝟏₂)
            + 8ν * 𝐞ʳ ⊗ˢ 𝟏₂ ⊗ˢ 𝐞ʳ - 8𝐞ʳ ⊗ 𝐞ʳ ⊗ 𝐞ʳ ⊗ 𝐞ʳ
    )
)

tsimplify(ℾ - ℾ₂)

2×2×2×2 TensND.TensRotated{4, 2, Sym{PyCall.PyObject}, Tensors.SymmetricTensor{4, 2, Sym{PyCall.PyObject}, 9}}:
[:, :, 1, 1] =
 0  0
 0  0

[:, :, 2, 1] =
 0  0
 0  0

[:, :, 1, 2] =
 0  0
 0  0

[:, :, 2, 2] =
 0  0
 0  0

Identically zero: the operator route and the closed form agree.

## Contraction with the stiffness

$\mathbb{k}=\mathbb{\Gamma}:\mathbb{C}$ is the object entering the
Lippmann–Schwinger equation of micromechanics.

In [5]:
ℂ₂ = 2λ * 𝕁₂ + 2μ * 𝕀₂
𝕜 = tsimplify(ℾ ⊡ ℂ₂)
get_array(𝕜)[1, 1, 1, 1]

   3 - 2⋅ν    
──────────────
     2        
4⋅π⋅r ⋅(ν - 1)

## Three dimensions

$$
\boldsymbol{G}=\frac{1}{16\pi\mu(1-\nu)r}
\Bigl((3-4\nu)\boldsymbol{1}+\underline{e}^r\otimes\underline{e}^r\Bigr)
$$

equivalently written with the bulk modulus, and the two forms must agree:

In [6]:
Spherical = coorsys_spherical()
θs, ϕs, rs = getcoords(Spherical)
𝐞ᶿ, 𝐞ᵠ, 𝐞ʳˢ = unitvec(Spherical)
ℬˢ = normalized_basis(Spherical)
@set_coorsys Spherical

𝕀, 𝕁, 𝕂 = iso_projectors(Val(3), Val(Sym))
𝟏 = tens_Id2(Val(3), Val(Sym))

𝐆₃ = 1 / (8PI * μ * (3k + 4μ) * rs) * ((3k + 7μ) * 𝟏 + (3k + μ) * 𝐞ʳˢ ⊗ 𝐞ʳˢ)
𝐆₃ᵥ = 1 / (16PI * μ * (1 - ν) * rs) * ((3 - 4ν) * 𝟏 + 𝐞ʳˢ ⊗ 𝐞ʳˢ)

tsimplify(𝐆₃ - 𝐆₃ᵥ)

3×3 TensND.TensRotated{2, 3, Sym{PyCall.PyObject}, Tensors.SymmetricTensor{2, 3, Sym{PyCall.PyObject}, 6}}:
 0  0  0
 0  0  0
 0  0  0

The same construction in 3-D:

In [7]:
HG₃ = -tsimplify(HESS(𝐆₃))
aHG₃ = get_array(HG₃)
𝕄₃ = SymmetricTensor{4, 3}((i, j, k, l) -> (aHG₃[i, k, j, l] + aHG₃[j, k, i, l] + aHG₃[i, l, j, k] + aHG₃[j, l, i, k]) / 4)
ℾ₃ = tsimplify(Tens(𝕄₃, ℬˢ))

ℾ₃ᶜ = tsimplify(
    1 / (16PI * μ * (1 - ν) * rs^3) * (
        -3𝕁 + 2(1 - 2ν) * 𝕀 + 3(𝟏 ⊗ 𝐞ʳˢ ⊗ 𝐞ʳˢ + 𝐞ʳˢ ⊗ 𝐞ʳˢ ⊗ 𝟏)
            + 12ν * 𝐞ʳˢ ⊗ˢ 𝟏 ⊗ˢ 𝐞ʳˢ - 15𝐞ʳˢ ⊗ 𝐞ʳˢ ⊗ 𝐞ʳˢ ⊗ 𝐞ʳˢ
    )
)

tsimplify(ℾ₃ - ℾ₃ᶜ)

3×3×3×3 TensND.TensRotated{4, 3, Sym{PyCall.PyObject}, Tensors.SymmetricTensor{4, 3, Sym{PyCall.PyObject}, 36}}:
[:, :, 1, 1] =
 0  0  0
 0  0  0
 0  0  0

[:, :, 2, 1] =
 0  …                                                                                          0
 0     -(ν + 1)*(4*ν - 3)*(sin(2*θ)*tan(θ) + cos(2*θ) - 1)/(32*pi*E*r^3*(ν - 1)*sin(θ)^2*tan(θ))
 0                                                                                             0

[:, :, 3, 1] =
 0  …  0
 0     0
 0     0

[:, :, 1, 2] =
 0  …                                                                                          0
 0     -(ν + 1)*(4*ν - 3)*(sin(2*θ)*tan(θ) + cos(2*θ) - 1)/(32*pi*E*r^3*(ν - 1)*sin(θ)^2*tan(θ))
 0                                                                                             0

[:, :, 2, 2] =
                                                                                         0  …  -(ν + 1)*(4*ν - 3)*(sin(2*θ)*tan(θ) + cos(2*θ) - 1)/(16*pi*E*r^3*(ν - 1)*sin(

Note the pattern: the 2-D coefficients $(-2,\,2,\,2,\,8,\,-8)$ become
$(-3,\,2,\,3,\,12,\,-15)$ in 3-D, and $r^{-2}$ becomes $r^{-3}$.

## The Jacobian of the induced deformation

For a unit point force along $\underline{e}_1$, the deformation gradient is
$\boldsymbol{1}+F\,\nabla(\boldsymbol{G}\cdot\underline{e}_1)$ and its
determinant measures the local volume change. Evaluated near the equator
$\theta=\pi/2$, $\varphi=0$:

In [8]:
Cartesian = coorsys_cartesian(symbols("x y z", real = true))
𝐞₁, 𝐞₂, 𝐞₃ = unitvec(Cartesian)
F = symbols("F", real = true)

J = tsimplify(det(𝟏 + F * GRAD(𝐆₃ ⋅ 𝐞₁)))
factor(tsimplify(subs(J, θs => PI / 2, ϕs => 0)))

0

---

*This notebook was generated using [Literate.jl](https://github.com/fredrikekre/Literate.jl).*